# 01a — Kalibrasi Kamera Sesuai Paper

Bagian ini **hanya** menangani kalibrasi kamera: mencari titik sudut papan catur,
mengoptimalkan subpiksel, mengurutkan 49 titik, lalu menghitung parameter kamera.

**Masukan:** folder berisi citra papan catur (`calib_XXXX.png`, grayscale 1280x720).
Papan: grid 8x8 kotak 20 mm (7x7 titik sudut dalam) + 3 lingkaran penanda
(origin, sumbu X, sumbu Y).

**Keluaran:**
1. Matriks intrinsik `K` dan koefisien distorsi `D`
2. Reprojection error (RMSE) per citra dan keseluruhan
3. Visualisasi tahapan pipeline (a)-(f)
4. `output/camera_params.npz` + `output/reprojection_rmse.csv` — dipakai oleh
   notebook `02_machine_calibration/kalibrasi_mesin.ipynb`

> Hasil eksekusi notebook asli sengaja dikosongkan (Clear All Outputs) supaya
> berkasnya ringan untuk GitHub. Isi original tetap utuh di
> `Semester 7/SEMPRO/Code/kalibrasicamera/kalibrasisesuaipaper.ipynb`.


In [ ]:
#KONFIFURASI PARAMETER

# ==========================================================================
# MUAT KONFIGURASI PATH  (satu sumber untuk semua notebook: config_paths.py)
# Semua path dataset ada di file itu. Ubah di sana, bukan di sini.
# ==========================================================================
import os
import sys

_p = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_p, "config_paths.py")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError(
            "config_paths.py tidak ditemukan. Jalankan notebook ini dari dalam folder "
            "repo hasil penataan, atau set environment KALIBRASI_REPO ke akar repo."
        )
    _p = _parent
sys.path.insert(0, _p)
import config_paths as CFG

CFG.diagnose()
print()

# Fungsi pipeline (EDLines, screening, subpixel, sorting) ada di
# calibration_pipeline.py supaya dipakai bersama oleh semua notebook.
from calibration_pipeline import (
    detect_edlines_raw, get_initial_corners_strict_paper,
    optimize_subpixel_gradient, sort_corners_robust_paper,
)


import cv2
import numpy as np
import glob
import os
from scipy.spatial import distance
import matplotlib.pyplot as plt

# ==========================================
# 1. KONFIGURASI PARAMETER
# ==========================================
CHESSBOARD_SIZE = CFG.CHESSBOARD_SIZE
SQUARE_SIZE = CFG.SQUARE_SIZE_MM  # mm
FOLDER_PATH = CFG.CAMERA_CALIB_DATASET
# Dataset alternatif (kalibrasi kamera sesi measurement error):
# FOLDER_PATH = CFG.resolve("measurement_error/x_camera_calibration")
objp = np.zeros((CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

vis_images = {}
image_files = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG', '*.JPEG'):
    image_files.extend(glob.glob(os.path.join(FOLDER_PATH, ext)))
image_files = sorted(list(set(image_files)))

if not image_files:
    print("ERROR: Tidak ada gambar ditemukan.")
else:
    sample_idx = 0 if len(image_files) > 7 else 2 
    sample_img = cv2.imread(image_files[sample_idx])
    sample_gray = cv2.cvtColor(sample_img, cv2.COLOR_BGR2GRAY)
    gray_3c = cv2.cvtColor(sample_gray, cv2.COLOR_GRAY2RGB)
    vis_images['a'] = gray_3c.copy()

In [ ]:
# 2. EDLINES DETECTOR
# ==========================================


sample_lines = detect_edlines_raw(sample_gray)

img_b = gray_3c.copy()
for line in sample_lines:
    cv2.line(img_b, (int(line[0]), int(line[1])), (int(line[2]), int(line[3])), (255, 0, 0), 2)

# ---> PASTIKAN BARIS INI ADA UNTUK MENYIMPAN GAMBAR KE CELL 8 <---
vis_images['b'] = img_b 

# Visualisasi
plt.figure(figsize=(8, 6))
plt.imshow(img_b)
plt.title('(b) Image after straight-line detection by EDLines', loc='left', fontweight='bold')
plt.axis('off')
plt.show()

In [ ]:
# 3. LINE SCREENING & INITIAL CORNERS
# ==========================================








sample_initial, sample_filt_lines = get_initial_corners_strict_paper(sample_gray, sample_lines)

img_c = gray_3c.copy()
for line in sample_filt_lines:
    cv2.line(img_c, (int(line[0]), int(line[1])), (int(line[2]), int(line[3])), (255, 0, 0), 2)
vis_images['c'] = img_c

img_d = gray_3c.copy()
for pt in sample_initial:
    cv2.circle(img_d, (int(pt[0]), int(pt[1])), 6, (255, 0, 0), -1)
vis_images['d'] = img_d

# Visualisasi
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
axs[0].imshow(img_c)
axs[0].set_title('(c) Result after straight-line screening', loc='left', fontweight='bold')
axs[0].axis('off')

axs[1].imshow(img_d)
axs[1].set_title('(d) Initial coordinate results of corner points', loc='left', fontweight='bold')
axs[1].axis('off')
plt.show()

In [ ]:
# 4. SUBPIXEL OPTIMIZATION
# ==========================================


sample_refined = optimize_subpixel_gradient(sample_gray, sample_initial)

img_e = gray_3c.copy()
if sample_refined is not None:
    for pt in sample_refined:
        # Perubahan: Gambar titik biru kecil, kemudian lingkaran merah lebih besar di sekitarnya
        cv2.circle(img_e, (int(pt[0]), int(pt[1])), 2, (255, 0, 0), -1) # Titik biru kecil
        cv2.circle(img_e, (int(pt[0]), int(pt[1])), 8, (0, 0, 255), 2)  # Lingkaran merah, lebih besar, garis lebih tebal, warna merah
vis_images['e'] = img_e

# Visualisasi
plt.figure(figsize=(8, 6))
plt.imshow(img_e)
plt.title('(e) Result of corner points subpixel optimization', loc='left', fontweight='bold')
plt.axis('off')
plt.show()

In [ ]:
# 5. ROBUST CORNER SORTING (FIXED ALIGNMENT)
# ==========================================


sample_sorted = sort_corners_robust_paper(sample_refined, sample_gray, CHESSBOARD_SIZE)

img_f = gray_3c.copy()
if sample_sorted is not None:
    for idx, pt in enumerate(sample_sorted): 
        cv2.circle(img_f, (int(pt[0]), int(pt[1])), 2, (255, 0, 0), -1)
        cv2.putText(img_f, str(idx+1), (int(pt[0])+4, int(pt[1])-4), 1, 1.5, (255, 0, 0), 2)
else:
    cv2.putText(img_f, "EDCIRCLES / SORTING GAGAL", (50, 100), 1, 2, (255, 255, 0), 2)
vis_images['f'] = img_f

# Visualisasi
plt.figure(figsize=(8, 6))
plt.imshow(img_f)
plt.title('(f) Result after corner sorting (Corrected Alignment)', loc='left', fontweight='bold')
plt.axis('off')
plt.show()

In [ ]:
# 6. FINAL PIPELINE: LOOP SEMUA GAMBAR & KALIBRASI
# ==========================================
objpoints, imgpoints, valid_filenames = [], [], []
img_shape = None
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

print(f"\nMemproses {len(image_files)} gambar...")

for idx, fname in enumerate(image_files):
    img = cv2.imread(fname)
    if img is None: continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if img_shape is None: img_shape = gray.shape[::-1]
    
    lines = detect_edlines_raw(gray)
    initial_corners, _ = get_initial_corners_strict_paper(gray, lines)
    
    if len(initial_corners) >= expected_pts:
        refined_corners = optimize_subpixel_gradient(gray, initial_corners)
        sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
        
        if sorted_corners is not None:
            objpoints.append(objp)
            imgpoints.append(sorted_corners.reshape(-1, 1, 2))
            valid_filenames.append(os.path.basename(fname))
            print(f"[{idx+1:02d}/{len(image_files):02d}] OK    - {os.path.basename(fname)}")
        else:
            print(f"[{idx+1:02d}/{len(image_files):02d}] GAGAL - {os.path.basename(fname)} (Sorting Error)")
    else:
        print(f"[{idx+1:02d}/{len(image_files):02d}] GAGAL - {os.path.basename(fname)} (Kurang titik awal)")

# ==========================================
# 7. KALIBRASI & EVALUASI REPROJECTION ERROR (RMSE) 
# ==========================================
if len(objpoints) > 0:
    print(f"Menghitung parameter kamera dari {len(objpoints)} gambar valid...")
    
    # 1. Hitung Parameter Kamera (Intrinsik & Distorsi)
    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, img_shape, None, None
    )

    print("\n--- 1. MATRIKS INTRINSIK (K) ---")
    print("Format: [[fx, 0, cx], [0, fy, cy], [0, 0, 1]]")
    print(np.round(camera_matrix, 4))
    
    print("\n--- 2. KOEFISIEN DISTORSI ---")
    print("Format: [k1, k2, p1, p2, k3]")
    print(np.round(dist_coeffs.ravel(), 5))

    # 2. Hitung Reprojection Error (RMSE) per Gambar Sesuai Persamaan 2.6
    print("\n--- 3. REPROJECTION ERROR (RMSE) PER DATA ---")
    total_sse = 0       # Sum of Squared Errors keseluruhan
    total_points = 0    # Total titik keseluruhan (N)
    
    for i in range(len(objpoints)):
        # Proyeksikan kembali titik 3D ke citra menggunakan parameter yang didapat (p_topi)
        imgpts2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], camera_matrix, dist_coeffs)
        
        # Hitung jarak Euclidean (Norm L2) antara titik observasi (p) dan proyeksi (p_topi)
        # cv2.norm dengan NORM_L2 menghasilkan: akar( jumlah( (p - p_topi)^2 ) )
        norm_l2 = cv2.norm(imgpoints[i], imgpts2, cv2.NORM_L2)
        
        # Kuadratkan norm_l2 untuk menghilangkan akar, mendapatkan jumlah kuadrat error (SSE)
        sse_per_image = norm_l2 ** 2
        n_points_per_image = len(imgpts2)
        
        # Hitung RMSE per gambar: akar ( SSE / n )
        rmse_per_image = np.sqrt(sse_per_image / n_points_per_image)
        
        # Akumulasi untuk menghitung rata-rata keseluruhan
        total_sse += sse_per_image
        total_points += n_points_per_image
        
        print(f"{valid_filenames[i]:<25} : {rmse_per_image:.4f} piksel")
        
    # Hitung keseluruhan RMSE (Root Mean Square Error) untuk semua gambar
    overall_rmse = np.sqrt(total_sse / total_points)
    
    # 3. Analisis Akurasi 
    print("\n--- 4. KESIMPULAN AKURASI ---")
    print(f"=> Keseluruhan Reprojection Error (RMSE) : {overall_rmse:.4f} piksel")
    
    if overall_rmse < 0.05:
        print("\n[STATUS]: SANGAT AKURAT!")
        print("Sesuai dengan target akurasi di mana error < 0.05 piksel.")
    elif overall_rmse < 0.15:
        print("\n[STATUS]: AKURAT.")
        print("Memenuhi standar akurasi tinggi untuk kalibrasi kamera.")
    else:
        print("\n[STATUS]: NORMAL.")
        print("Kalibrasi berhasil, namun pastikan dataset gambar tidak ada yang blur atau terlalu gelap.")

else:
    print("Kalibrasi dibatalkan: Tidak ada data valid untuk dikalibrasi.")

In [ ]:
# 8. TAMPILKAN VISUALISASI PIPELINE (GAMBAR SAMPLE)
# ==========================================
titles = {
    'a': '(a) Original grayscale image',
    'b': '(b) Image after straight-line detection by EDLines',
    'c': '(c) Result after straight-line screening',
    'd': '(d) Initial coordinate results of corner points',
    'e': '(e) Result of corner points subpixel optimization',
    'f': '(f) Result after corner sorting'
}

print("Menampilkan visualisasi dalam bentuk Grid (Gabungan)...")

# --- OPSI 1: TAMPILAN GRID SUBPLOT (SANGAT DIREKOMENDASIKAN UNTUK SKRIPSI) ---
fig, axs = plt.subplots(2, 3, figsize=(20, 12))
axs = axs.ravel() # Meratakan array axis agar mudah di-loop

for i, key in enumerate(['a', 'b', 'c', 'd', 'e', 'f']):
    # Konversi BGR ke RGB untuk matplotlib jika gambar berwarna
    # (Gambar vis_images kita sudah dalam format RGB dari Cell 1)
    axs[i].imshow(vis_images[key])
    axs[i].set_title(titles[key], loc='left', fontweight='bold', fontsize=14)
    axs[i].axis('off')

plt.tight_layout()
plt.show()

print("\nMenampilkan visualisasi secara individual...")

# --- OPSI 2: TAMPILAN INDIVIDUAL (SEPERTI KODE ASLI) ---
for key in ['a', 'b', 'c', 'd', 'e', 'f']:
    plt.figure(figsize=(8, 6))
    plt.imshow(vis_images[key])
    plt.title(titles[key], loc='left', fontweight='bold')
    plt.axis('off')
    plt.show()

In [ ]:
# --- OPSI 2: TAMPILAN INDIVIDUAL (TANPA JUDUL & FULL SIZE) ---
# Bagus jika Anda ingin menyimpan gambar satu per satu tanpa margin putih
print("\nMenampilkan visualisasi secara individual (Full Frame, No Title)...")

for key in ['a', 'b', 'c', 'd', 'e', 'f']:
    if key in vis_images:
        img = vis_images[key]
        h, w = img.shape[:2]
        
        # Penyesuaian DPI agar ukuran figure sesuai dengan resolusi gambar
        dpi = 100
        fig = plt.figure(figsize=(w/dpi, h/dpi))
        
        # add_axes([left, bottom, width, height]) 
        # Menggunakan [0,0,1,1] untuk menghapus semua margin putih/padding
        ax = fig.add_axes([0, 0, 1, 1])
        
        ax.imshow(img)
        ax.axis('off')
        plt.show()
    else:
        print(f"Peringatan: Gambar untuk key '{key}' tidak ditemukan di vis_images.")

In [ ]:
# === [EKSPOR] PARAMETER KAMERA UNTUK KALIBRASI MESIN ===
# Menyimpan matriks intrinsik dan koefisien distorsi sebagai CSV agar bisa dibaca
# notebook 02_machine_calibration/kalibrasi_mesin.ipynb (dan skrip lain) tanpa
# menjalankan ulang kalibrasi kamera.
import os
import numpy as np
import pandas as pd

OUTPUT_DIR = os.path.join(CFG.OUTPUT_DIR, "camera_calibration")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Matriks intrinsik K (3x3) -- numerik murni, mudah dibaca np.loadtxt
np.savetxt(os.path.join(OUTPUT_DIR, "camera_matrix.csv"), camera_matrix,
           delimiter=",", fmt="%.10f")

# 2. Koefisien distorsi [k1, k2, p1, p2, k3] -- satu baris
np.savetxt(os.path.join(OUTPUT_DIR, "dist_coeffs.csv"),
           np.asarray(dist_coeffs).reshape(1, -1), delimiter=",", fmt="%.10f")

# 3. Keterangan dalam bentuk tabel parameter/nilai (untuk dibaca manusia)
row = {"image_width": int(img_shape[0]), "image_height": int(img_shape[1]),
       "fx": camera_matrix[0, 0], "fy": camera_matrix[1, 1],
       "cx": camera_matrix[0, 2], "cy": camera_matrix[1, 2]}
for i, k in enumerate(np.asarray(dist_coeffs).ravel()):
    row[f"d{i + 1}"] = float(k)
pd.DataFrame(sorted(row.items()), columns=["parameter", "value"]).to_csv(
    os.path.join(OUTPUT_DIR, "camera_params.csv"), index=False)

# 4. RMSE reprojection per citra (Persamaan 2.6 pada naskah)
rows = []
for _i in range(len(objpoints)):
    _imgpts2, _ = cv2.projectPoints(
        objpoints[_i], rvecs[_i], tvecs[_i], camera_matrix, dist_coeffs)
    _n2 = cv2.norm(imgpoints[_i], _imgpts2, cv2.NORM_L2)
    rows.append((valid_filenames[_i], float(np.sqrt((_n2 ** 2) / len(_imgpts2)))))
df_rmse = pd.DataFrame(rows, columns=["file", "rmse_px"])
df_rmse.to_csv(os.path.join(OUTPUT_DIR, "reprojection_rmse.csv"), index=False)

print(f"Parameter kamera disimpan sebagai CSV di:\n  {OUTPUT_DIR}")
for f in ["camera_matrix.csv", "dist_coeffs.csv", "camera_params.csv", "reprojection_rmse.csv"]:
    print(f"  - {f}")
print(f"\n{len(rows)} citra valid | RMSE reprojection rata-rata: "
      f"{df_rmse['rmse_px'].mean():.4f} px (min {df_rmse['rmse_px'].min():.4f}, "
      f"max {df_rmse['rmse_px'].max():.4f})")
